In [29]:
import json
import math
import random
import time
import os
from datetime import datetime, timezone

import pandas as pd
import redis
from kafka import KafkaProducer
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, to_timestamp, window,
    avg, stddev_pop, sum as spark_sum, count
)
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

print("Libraries imported successfully!")

Libraries imported successfully!


In [30]:
KAFKA_BOOTSTRAP = "localhost:9092"
KAFKA_TOPIC = "sales-transactions"
REDIS_HOST = "localhost"
REDIS_PORT = 6379

SERVICE_LEVEL_Z = 1.65
LEAD_TIME_BUCKETS = 4

CHECKPOINT_DIR = os.path.abspath("checkpoint/week9")

PRODUCTS = ["PROD001", "PROD002", "PROD003", "PROD004"]

schema = StructType([
    StructField("product_id", StringType(), False),
    StructField("quantity", IntegerType(), False),
    StructField("timestamp", StringType(), False)
])

print("Configuration completed!")

Configuration completed!


In [31]:
def generate_transaction():
    return {
        "product_id": random.choice(PRODUCTS),
        "quantity": random.randint(1, 12),
        "timestamp": datetime.now(timezone.utc).isoformat()
    }

pd.DataFrame([generate_transaction() for _ in range(10)])

,product_id,quantity,timestamp
0,PROD003,2,2026-10-27T06:22:40.204797+00:00
1,PROD003,12,2026-10-27T06:22:40.204797+00:00
2,PROD004,11,2026-10-27T06:22:40.204797+00:00
3,PROD002,12,2026-10-27T06:22:40.204797+00:00
4,PROD004,10,2026-10-27T06:22:40.204797+00:00
5,PROD002,5,2026-10-27T06:22:40.204797+00:00
6,PROD001,3,2026-10-27T06:22:40.204797+00:00
7,PROD001,8,2026-10-27T06:22:40.204797+00:00
8,PROD003,5,2026-10-27T06:22:40.204797+00:00
9,PROD001,1,2026-10-27T06:22:40.204797+00:00


In [32]:
def publish_transactions(number_of_events=60, delay=1):
    producer = KafkaProducer(
        bootstrap_servers=KAFKA_BOOTSTRAP,
        value_serializer=lambda x: json.dumps(x).encode("utf-8"),
        key_serializer=lambda x: x.encode("utf-8")
    )

    try:
        for i in range(number_of_events):
            event = generate_transaction()

            producer.send(
                KAFKA_TOPIC,
                key=event["product_id"],
                value=event
            )
            producer.flush()

            print(f"Transaction {i+1}: {event}")
            time.sleep(delay)

    finally:
        producer.close()

print("Kafka producer ready!")

Kafka producer ready!


In [33]:
import os
import sys
import shutil
import subprocess

print("Python version:", sys.version)
print("Python executable:", sys.executable)
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("Java executable:", shutil.which("java"))

try:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        timeout=15
    )
    print("\nJava version details:")
    print(result.stderr or result.stdout)
    print("Java exit code:", result.returncode)
except Exception as error:
    print("\nJava error:", repr(error))

Python version: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
Python executable: C:\Users\gokul\AppData\Local\Microsoft\WindowsApps\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\python.exe
JAVA_HOME: None
Java executable: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE

Java version details:
java version "26.0.1" 2026-04-21
Java(TM) SE Runtime Environment (build 26.0.1+8-34)
Java HotSpot(TM) 64-Bit Server VM (build 26.0.1+8-34, mixed mode, sharing)

Java exit code: 0


In [34]:
import sys
import os
import shutil
import subprocess
import importlib.metadata as metadata

print("Python:", sys.version)
print("PySpark version:", metadata.version("pyspark"))
print("JAVA_HOME:", os.environ.get("JAVA_HOME"))
print("Java path:", shutil.which("java"))

try:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print("Java output:")
    print(result.stdout)
    print(result.stderr)
except Exception as e:
    print("Java error:", e)

Python: 3.11.9 (tags/v3.11.9:de54cf5, Apr  2 2024, 10:12:12) [MSC v.1938 64 bit (AMD64)]
PySpark version: 4.2.0
JAVA_HOME: None
Java path: C:\Program Files\Common Files\Oracle\Java\javapath\java.EXE
Java output:

java version "26.0.1" 2026-04-21
Java(TM) SE Runtime Environment (build 26.0.1+8-34)
Java HotSpot(TM) 64-Bit Server VM (build 26.0.1+8-34, mixed mode, sharing)



In [35]:
import subprocess

try:
    result = subprocess.run(
        ["java", "-version"],
        capture_output=True,
        text=True,
        timeout=10
    )
    print("Return code:", result.returncode)
    print(result.stderr or result.stdout)
except Exception as e:
    print(repr(e))

Return code: 0
java version "26.0.1" 2026-04-21
Java(TM) SE Runtime Environment (build 26.0.1+8-34)
Java HotSpot(TM) 64-Bit Server VM (build 26.0.1+8-34, mixed mode, sharing)



In [36]:
demand_stats = (
    transactions
    .groupBy(
        window(
            col("event_time"),
            "1 minute",
            "30 seconds"
        ),
        col("product_id")
    )
    .agg(
        spark_sum("quantity").alias("demand_total"),
        avg(col("quantity").cast("double")).alias("avg_demand"),
        stddev_pop(
            col("quantity").cast("double")
        ).alias("stddev_demand"),
        count("*").alias("transaction_count")
    )
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "product_id",
        "demand_total",
        "avg_demand",
        "stddev_demand",
        "transaction_count"
    )
)

print("Demand statistics calculation ready!")

NameError: name 'transactions' is not defined

In [ ]:
def update_redis(batch_df, batch_id):
    if batch_df.rdd.isEmpty():
        return

    client = redis.Redis(
        host=REDIS_HOST,
        port=REDIS_PORT,
        decode_responses=True
    )

    try:
        for row in batch_df.collect():
            avg_demand = float(row["avg_demand"] or 0)
            std_demand = float(row["stddev_demand"] or 0)

            safety_stock = math.ceil(
                SERVICE_LEVEL_Z
                * std_demand
                * math.sqrt(LEAD_TIME_BUCKETS)
            )

            reorder_point = math.ceil(
                avg_demand * LEAD_TIME_BUCKETS
                + safety_stock
            )

            result = {
                "product_id": row["product_id"],
                "demand_total": str(row["demand_total"]),
                "avg_demand": f"{avg_demand:.2f}",
                "stddev_demand": f"{std_demand:.2f}",
                "safety_stock": str(safety_stock),
                "reorder_point": str(reorder_point),
                "transaction_count": str(row["transaction_count"]),
                "batch_id": str(batch_id)
            }

            client.hset(
                f"safety-stock:{row['product_id']}",
                mapping=result
            )

            print(
                f"Product: {row['product_id']} | "
                f"Safety Stock: {safety_stock} | "
                f"Reorder Point: {reorder_point}"
            )

    finally:
        client.close()

print("Redis update function ready!")

In [ ]:
stream_query = (
    demand_stats.writeStream
    .outputMode("update")
    .foreachBatch(update_redis)
    .option("checkpointLocation", CHECKPOINT_DIR)
    .trigger(processingTime="10 seconds")
    .start()
)

print("Real-time streaming started!")

In [ ]:
publish_transactions(
    number_of_events=60,
    delay=1
)

In [ ]:
client = redis.Redis(
    host=REDIS_HOST,
    port=REDIS_PORT,
    decode_responses=True
)

keys = sorted(client.scan_iter("safety-stock:*"))

if not keys:
    print("No results found. Check Kafka and Spark.")

for key in keys:
    print("\n", key)

    data = client.hgetall(key)

    for field, value in data.items():
        print(f"{field}: {value}")

client.close()

In [ ]:
if stream_query.isActive:
    stream_query.stop()

spark.stop()

print("Streaming stopped successfully!")